In [0]:
%pip install yfinance
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import yfinance as yf
import pandas as pd
from datetime import datetime,timedelta
from pyspark.sql.functions import lit, current_timestamp,col,lead,when
from pyspark.sql import Window



In [0]:
def read_date_control(path)->[str,str]:
    with open(path) as f:
        result=f.readline().strip()
        prev_date_string,curr_date_string=result.split("|")
    return datetime.strptime(prev_date_string, "%Y%m%d"),datetime.strptime(curr_date_string, "%Y%m%d")

In [0]:
def read_data_from_api(symbols,prev_date,curr_date)->pd.DataFrame:
    data = yf.download(tickers=" ".join(symbols),
                   start=prev_date+timedelta(days=1),
                   end=curr_date + timedelta(days=1),
                   interval="1d",
                   group_by="ticker")
    data.reset_index(inplace=True)
    rows = []
    for ticker in symbols:
        df_ticker = data[ticker].copy()
        df_ticker["Date"] = data["Date"]
        df_ticker["Ticker"] = ticker.split(".")[0]
        rows.append(df_ticker)

    # Concatenate into one DataFrame
    pandas_df = pd.concat(rows)
    pandas_df = pandas_df[["Ticker","Open","High","Low","Close","Volume","Date"]]
    return pandas_df

In [0]:
def convert_to_sparkdf(pandas_df)->"spark dataframe":
    # Define schema
    schema = StructType([
        StructField("Ticker", StringType(), True),
        StructField("Open", DoubleType(), True),
        StructField("High", DoubleType(), True),
        StructField("Low", DoubleType(), True),
        StructField("Close", DoubleType(), True),
        StructField("Volume", LongType(), True),
        StructField("Date", DateType(), True),
    ])
    df = spark.createDataFrame(pandas_df, schema=schema)
    return df

In [0]:


def scd2_etl(staging_df, target_table):
    """
    Apply SCD Type-2 ETL:
    - If target table has no rows → Day0 load (insert all rows with eff_start_date).
    - If target table has rows → DayN merge (CDC updates).
    """
    cur_ts=datetime.now()
    active_df=spark.read.table("default.dim_stock")\
                        .filter(col('is_active')==True)
                    
    if active_df.count()==0:
        print(f"Table {target_table} is empty. Running Day0 load...")
    else:
        print(f"Table {target_table} has {active_df.count()} rows. Running DayN merge..")
    staging_df=staging_df.withColumnRenamed('Date','eff_start_date')\
                        .withColumn('eff_end_date',lit(None).cast('date'))\
                        .withColumn('dw_load_tmp',lit(cur_ts).cast('timestamp'))\
                        .withColumn('dw_upd_load_tmp',lit(cur_ts).cast("timestamp"))\
                        .withColumn('is_active',lit(True))\
                        .withColumn('flag',lit('delta'))

    active_df=active_df.withColumn('flag',lit('snapshot'))

    snap_delta=staging_df.unionByName(active_df)
    
    w = Window.partitionBy("Ticker").orderBy("eff_start_date")
    #need some insert twick for upd  but going with this during every load even if no change is there this will update wil change

    day0_df1 = snap_delta.withColumn("eff_end_date", lead("eff_start_date").over(w))\
                        .withColumn("dw_upd_load_tmp", when(col('flag')=='snapshot',lit(cur_ts).cast('timestamp')).otherwise(col("dw_upd_load_tmp")))\
                        .drop('flag')

    # Mark last record per ticker as active
    lrf = day0_df1.withColumn("is_active",(col("eff_end_date").isNull()).cast("boolean"))
    

    inactive=lrf.filter(col('is_active')==False)
    inactive.write.format("delta").mode("append").saveAsTable(target_table)

    active=lrf.filter(col('is_active')==True)
    active.write.format("delta").mode("overwrite").option("replaceWhere", "is_active = true") \
                .saveAsTable(target_table)
 
    print("load completed.")
    


In [0]:

date_control_path="/Workspace/Users/vigneshgv2001@gmail.com/stock_analysis_Databricks_ETL/stock_date_control.csv"

symbols = [
    "RELIANCE.NS","INFY.NS","TCS.NS","HDFCBANK.NS","SBIN.NS",
    "ICICIBANK.NS","AXISBANK.NS","KOTAKBANK.NS","ITC.NS","LT.NS",
    "BHARTIARTL.NS","SUNPHARMA.NS","ONGC.NS","ADANIGREEN.NS","ADANIPORTS.NS",
    "HINDUNILVR.NS","BAJFINANCE.NS","MARUTI.NS","WIPRO.NS","TECHM.NS"
]

prev_date,curr_date=read_date_control(date_control_path)

pandas_df=read_data_from_api(symbols,prev_date,curr_date)
#print(pandas_df)
df=convert_to_sparkdf(pandas_df)

scd2_etl(staging_df=df,target_table="default.dim_stock")



In [0]:

'''from IPython.core.dispaly import HTML
display(HTML("<style>"+
             "pre{white-space:pre !important;overflow:visible;}"+
             "div #notebook-containr{width:90%}"+
             "div.prompt:empty{height:95%}"
             "</style>"))'''